In [ ]:
# ================================
# MUFFIN vs CHIHUAHUA - CNN MODEL
# FULL KAGGLE NOTEBOOK CODE
# ================================

import numpy as np
import pandas as pd
import os
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ================================
# PATHS (KAGGLE)
# ================================
TRAIN_DIR = '/kaggle/input/muffin-vs-chihuahua-image-classification/train'
TEST_DIR = '/kaggle/input/muffin-vs-chihuahua-image-classification/test'

# ================================
# PARAMETERS
# ================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ================================
# DATA GENERATORS
# ================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# ================================
# CNN MODEL
# ================================
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# ================================
# COMPILE
# ================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ================================
# CALLBACKS
# ================================
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

# ================================
# TRAIN
# ================================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)

# ================================
# TEST DATA
# ================================
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode=None,
    shuffle=False
)

# ================================
# PREDICTION
# ================================
pred_probs = model.predict(test_generator)

# Convert probabilities to labels
labels = ['chihuahua' if p > 0.5 else 'muffin' for p in pred_probs.flatten()]

# Extract filenames
filenames = [os.path.basename(f) for f in test_generator.filenames]

# ================================
# SUBMISSION FILE
# ================================
submission = pd.DataFrame({
    'id': filenames,
    'label': labels
})

submission.to_csv('submission.csv', index=False)

print("✅ submission.csv created successfully!")
